In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp
from datetime import datetime
from dateutil.relativedelta import relativedelta

#Parametros de carga
dbutils.widgets.text("tipo_carga","Incremental")
dbutils.widgets.text("fecha_carga", "2025-06-30")
dbutils.widgets.text("meses", "2", "Meses")

tipo_carga = dbutils.widgets.get("tipo_carga")
fecha_carga = dbutils.widgets.get("fecha_carga")
meses = int(dbutils.widgets.get("meses"))

fecha_base = datetime.strptime(fecha_carga, "%Y-%m-%d").date().replace(day=1)
particiones = [(fecha_base - relativedelta(months=i)).strftime("%Y-%m-%d") for i in range(meses+1)]

if tipo_carga == "Historico":
    df_compras = spark.table("azdb_sales_store.linio.silver_compras")
    df_detalles = spark.table("azdb_sales_store.linio.silver_detalles")

    df_fact_compras = (
        df_compras.alias("a")
        .join(df_detalles.alias("b"), on="factura", how="inner")
        .select(
            col("a.periodo"),
            col("a.venta_id"),
            col("a.factura"),
            col("a.tipo_compra"),
            col("a.fecha_orden"),
            col("a.fecha_entrega"),
            col("a.fecha_envio"),
            col("a.estado"),
            col("a.cliente_id"),
            col("a.vendedor"),
            col("a.departamento"),
            col("a.metodo_pago"),
            col("a.grupo_dias_envio"),
            col("b.detalle_id"),
            col("b.producto_id"),
            col("b.unidades"),
            col("b.subtotal"),
            current_timestamp().alias("fecha_actualizacion")
        )
    )
    (
        df_fact_compras.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("azdb_sales_store.linio.gold_fact_compras")
    )

else:

    df_detalles = spark.table("azdb_sales_store.linio.silver_detalles")


    df_compras = (
        spark.table("azdb_sales_store.linio.silver_compras")
        .filter(col("periodo").isin(particiones))
    )

    df_fact_compras = (
        df_compras.alias("a")
        .join(df_detalles.alias("b"), on="factura", how="inner")
        .select(
            col("a.periodo"),
            col("a.venta_id"),
            col("a.factura"),
            col("a.tipo_compra"),
            col("a.fecha_orden"),
            col("a.fecha_entrega"),
            col("a.fecha_envio"),
            col("a.estado"),
            col("a.cliente_id"),
            col("a.vendedor"),
            col("a.departamento"),
            col("a.metodo_pago"),
            col("a.grupo_dias_envio"),
            col("b.detalle_id"),
            col("b.producto_id"),
            col("b.unidades"),
            col("b.subtotal"),
            current_timestamp().alias("fecha_actualizacion")
        )
    )
    (
    DeltaTable.forName(spark, "azdb_sales_store.linio.gold_fact_compras").alias("gold")
    .merge(
        df_fact_compras.alias("new"),
        "gold.detalle_id = new.detalle_id AND gold.periodo IN (" + ",".join([f"'{p}'" for p in particiones]) + ")"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

